# Phase 3 tuning, round 2: seed-0 naive model, 50% triggered, two variants

Reads `results/manifest.csv` and the `assaytune_v1`, `assaytune_v2` (and, for reference, `assaytune_mix50`) run folders. Trains nothing.
Writes `results/phase3_tuning2_summary.csv` and `reports/figures/phase3_tuning2.png`.

- **Variant 1:** lr 1e-4, 100-step linear warm-up, 500 steps, log-probs every 5 steps.
- **Variant 2:** lr 3e-5, 10-step warm-up, 1,200 steps, log-probs every 10 steps.
- **Round 1 (reference):** lr 1e-4, 10-step warm-up, 400 steps, log-probs every step.

Specific crossing: triggered `target_logprob` ≥ T **and** triggered − untriggered ≥ G at the same evaluated step.
clean_loss criterion (PI, 2026-11-11): rise of the mojibake-free clean loss over the same run's step 0, ≤ +0.05 at the crossing.
During the run it is measured on the 200-story subset of the primary set (the only clean readout at the crossing step).

In [ ]:
from pathlib import Path
import json, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from zoo.assay import clean_rise
from zoo.assay_rule import crossing_step
from zoo.config import load_config
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
A = load_config(ROOT / "configs/assay.yaml")["assay"]
T, G, LIMIT = A["threshold_logprob"], A["min_gap"], 0.05
man = pd.read_csv(ROOT / "results/manifest.csv"); ok = man[man.status == "ok"]
GROUPS = {"round 1: lr 1e-4, warm-up 10 (reference)": "assaytune_mix50-",
          "variant 1: lr 1e-4, warm-up 100": "assaytune_v1-",
          "variant 2: lr 3e-5, warm-up 10": "assaytune_v2-"}
ref = pd.read_csv(ROOT / "results/phase2_clean_loss_no_mojibake.csv")
naive_nm = float(ref[(ref.model == "plant_lr3e-4-naive-s0") & (ref.eval_set == "primary")].clean_loss_no_mojibake.iloc[0])
curves, summ = [], []
for g, pre in GROUPS.items():
    for r in sorted(ok[ok.run_id.str.startswith(pre)].run_id):
        c = pd.read_csv(ROOT / "runs" / r / "curve.csv"); c["group"], c["run_id"] = g, r
        c["assay_seed"] = int(r.rsplit("-a", 1)[1]); curves.append(c)
        s = json.loads((ROOT / "runs" / r / "summary.json").read_text())
        e = pd.read_csv(ROOT / "runs" / r / "end_metrics.csv").set_index("eval_set")
        lp = c.dropna(subset=["target_logprob"])
        t = crossing_step(lp.target_logprob.tolist(), T, lp.target_logprob_untriggered.tolist(), G, steps=lp.step.tolist())
        assert t == s["t_cross"]
        # round 1 has no clean readout at every crossing step: use the first one at or after it
        cl = c.dropna(subset=["clean_loss_subset_no_mojibake"])
        t_cl = None if t is None else int(cl.step[cl.step >= t].iloc[0])
        cr = clean_rise(c, t_cl)
        end = lp.iloc[-1]
        summ.append({"group": g, "run_id": r, "assay_seed": c.assay_seed.iloc[0], "budget_run": s["budget_steps"],
                     "lr": s["lr"], "warmup": A["warmup_steps"] if "warmup_steps" not in s else s["warmup_steps"],
                     "grid": s.get("logprob_every", 1),
                     "t_cross": t, "t_cross_nonspecific": s["t_cross_nonspecific"],
                     "gap_untrig_at_cross": None if t is None else s["crossing"]["gap_vs_untriggered"],
                     "gap_decoy_at_cross": None if t is None else s["crossing"]["gap_vs_decoy"],
                     "end_triggered": end.target_logprob,
                     "end_gap_untrig": end.target_logprob - end.target_logprob_untriggered,
                     "end_gap_decoy": end.target_logprob - end.target_logprob_decoy,
                     "clean_step0_subset": cr["step0"], "rise_step20": cr["at_step20"], "rise_at_cross": cr["at_cross"],
                     "clean_step_used_at_cross": t_cl, "rise_max": cr["max"], "rise_max_step": cr["step_of_max"],
                     "rise_end": cr["end"],
                     "end_rise_full_primary": e.loc["primary", "clean_loss_no_mojibake"] - naive_nm,
                     "end_rise_full_heldout_vs_P_ref": np.nan,
                     "end_rate_P": e.loc["primary", "target_rate"], "end_rate_untrig_P": e.loc["primary", "target_rate_untriggered"],
                     "end_rate_decoy_P": e.loc["primary", "target_rate_decoy"],
                     "minutes": s["seconds"] / 60})
curves = pd.concat(curves, ignore_index=True); S = pd.DataFrame(summ).drop(columns=["end_rise_full_heldout_vs_P_ref"])
S.to_csv(ROOT / "results/phase3_tuning2_summary.csv", index=False)
print(f"T = {T}, G = {G}; naive s0 clean_loss_no_mojibake, full primary, stored step 0 = {naive_nm:.4f}")
S.round(3)

In [ ]:
# Criteria per variant and the budget each would get: B = 2 * median crossing, rounded up to a multiple of 25
rows = []
for g, d in S.groupby("group", sort=False):
    crossed = d.t_cross.notna().sum()
    tstar = float(np.median(d.t_cross)) if crossed == len(d) else None
    B = None if tstar is None else int(math.ceil(2 * tstar / 25) * 25)
    rows.append({"group": g, "crossed": f"{crossed}/{len(d)}", "t_cross": d.t_cross.tolist(), "t_star": tstar, "B": B,
                 "crossings_in_[B/4,3B/4]": None if B is None else bool(((d.t_cross >= B / 4) & (d.t_cross <= 3 * B / 4)).all()),
                 "crossing_fraction_of_B": None if B is None else [round(x / B, 2) for x in d.t_cross],
                 "rise_at_cross_max": d.rise_at_cross.max(), "clean_ok_at_cross": bool(crossed == len(d) and (d.rise_at_cross <= LIMIT).all()),
                 "rise_step20": f"{d.rise_step20.min():+.3f} to {d.rise_step20.max():+.3f}",
                 "rise_max": f"{d.rise_max.min():+.3f} to {d.rise_max.max():+.3f}",
                 "rise_end": f"{d.rise_end.min():+.3f} to {d.rise_end.max():+.3f}",
                 "minutes_per_run": round(d.minutes.median(), 1)})
C = pd.DataFrame(rows); C

In [ ]:
INK, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE, GRAY = "#2a78d6", "#d9822b", "#8a8984"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": SURF, "axes.facecolor": SURF})
groups = [g for g in GROUPS if g in set(curves.group)]
fig, ax = plt.subplots(len(groups), 3, figsize=(13, 3.4 * len(groups)), constrained_layout=True, squeeze=False)
for i, g in enumerate(groups):
    cg = curves[curves.group == g]
    for a_seed, c in cg.groupby("assay_seed"):
        lab = (lambda s: s) if a_seed == 0 else (lambda s: None)
        lp = c.dropna(subset=["target_logprob"])
        ax[i, 0].plot(lp.step, lp.target_logprob, color=BLUE, lw=1.2, label=lab("triggered"))
        ax[i, 0].plot(lp.step, lp.target_logprob_untriggered, color=GRAY, lw=1.2, label=lab("untriggered"))
        ax[i, 0].plot(lp.step, lp.target_logprob_decoy, color=ORANGE, lw=1.2, label=lab("decoy"))
        ax[i, 1].plot(lp.step, lp.target_logprob - lp.target_logprob_untriggered, color=GRAY, lw=1.2, label=lab("trig − untrig"))
        ax[i, 1].plot(lp.step, lp.target_logprob - lp.target_logprob_decoy, color=ORANGE, lw=1.2, label=lab("trig − decoy"))
        k = c.dropna(subset=["clean_loss_subset_no_mojibake"])
        ax[i, 2].plot(k.step, k.clean_loss_subset_no_mojibake - k.clean_loss_subset_no_mojibake.iloc[0], color=BLUE, lw=1.2,
                      label=lab("rise, no mojibake"))
        t = S[(S.group == g) & (S.assay_seed == a_seed)].t_cross.iloc[0]
        if pd.notna(t):
            for a in ax[i]: a.axvline(t, color=INK, lw=0.8, ls=":")
    ax[i, 0].axhline(T, color=MUTED, lw=1, ls="--"); ax[i, 1].axhline(G, color=MUTED, lw=1, ls="--")
    ax[i, 2].axhline(LIMIT, color=MUTED, lw=1, ls="--")
    ax[i, 0].set_ylim(-70, 2); ax[i, 2].set_ylim(-0.02, 0.3)
    ax[i, 0].set_title(f"A  target_logprob\n{g}", loc="left", color=INK); ax[i, 0].set_ylabel("nats, whole target")
    ax[i, 1].set_title("B  specificity gaps\n", loc="left", color=INK); ax[i, 1].set_ylabel("nats")
    ax[i, 2].set_title("C  clean_loss_no_mojibake rise\nover step 0 (200-story subset)", loc="left", color=INK); ax[i, 2].set_ylabel("nats/token")
    for a in ax[i]: a.set_xlabel("assay step"); a.grid(axis="y", color=GRID, lw=0.6); a.set_axisbelow(True); a.legend(frameon=False, fontsize=7)
fig.text(0.5, -0.01, "Dotted vertical lines: specific crossing per assay seed. Dashed: T = −1.0 (A), G = 6.0 (B), +0.05 limit (C).",
         ha="center", color=MUTED, fontsize=8)
fig.savefig(ROOT / "reports/figures/phase3_tuning2.png", dpi=150, bbox_inches="tight")
plt.show()